In [ ]:
import sys
sys.path.append("..")   # to import from parent dir
import numpy as np
from IPython.display import HTML
import tonic
import matplotlib.pyplot as plt
import os.path as osp

from star8 import StarMovement
from utils.visualize_utils import animate_events
from utils.data_utils import *
from idn.loader.loader_dsec import HarrisRecursive

In [ ]:
# Image/grid size
img_height = 256
img_width = 256
image_size = (img_height, img_width)

center = np.array([img_width // 2, img_height // 2])

total_frames = 2_000

num_points = 5
outer_radius = 40
inner_radius = 10

number_of_rotations = 2

# Trajectory of the local shape
added_height = 200
start_pos = np.array([center[0], 0])

In [ ]:
import numpy as np
from matplotlib.path import Path

class StarMovementV2(StarMovement):
   
    def __init__(self, leg_angle_deg=30, **kwargs):
       
        """
        Initialize the star movement with a specified leg angle.

        Parameters:
        leg_angle_deg (float): The angle (in degrees) of the legs of the star shape.
        **kwargs: Additional arguments passed to the parent class.
        """
        self.leg_angle_deg = leg_angle_deg
        super().__init__(**kwargs)
       

    def define_shape(self):
        """
        Create a star-like shape centered at the origin,
        with legs of a given angle instead of a fixed inner radius.
        """
        # Convert to radians
        leg_angle = np.deg2rad(self.leg_angle_deg)
        num_points = self.num_points
        outer_r = self.outer_radius

        # The central angle between outer points
        delta_theta = 2 * np.pi / num_points

        # Compute the inner radius geometrically:
        # The inner radius forms where two outer points meet at the given leg_angle.
        # Formula derived from isosceles triangle geometry:
        inner_r = outer_r * np.sin(leg_angle / 2) / np.sin(delta_theta / 2 + leg_angle / 2)

        # Construct alternating angles for outer and inner vertices
        angles = np.linspace(0, 2 * np.pi, num_points * 2, endpoint=False)

        radii = np.empty_like(angles)
        radii[0::2] = outer_r      # outer points
        radii[1::2] = inner_r      # inner points

        # Compute coordinates
        x = radii * np.cos(angles)
        y = radii * np.sin(angles)
        vertices = np.vstack([x, y]).T

        # Close the shape
        codes = [Path.MOVETO] + [Path.LINETO] * (len(vertices) - 1) + [Path.CLOSEPOLY]
        vertices = np.vstack([vertices, vertices[0]])  # close polygon
        anchor_point = np.array([0, 0])

        return Path(vertices, codes), anchor_point


star8v2 = StarMovementV2(
    total_frames=total_frames,
    image_size=image_size,
    face_color="black",
    num_points=5,
    outer_radius=outer_radius,
    inner_radius=inner_radius,
    number_of_rotations=1,
    leg_angle_deg = 30,
)

# Display animation

anim = star8v2.create_animation(frame_step=20)
HTML(anim.to_jshtml())

In [ ]:
star8 = StarMovement(
    total_frames=total_frames,
    image_size=image_size,
    face_color="black",
    num_points=num_points,
    outer_radius=outer_radius,
    inner_radius=inner_radius,
    number_of_rotations=number_of_rotations,
)

# Display animation

anim = star8.create_animation(frame_step=20)
HTML(anim.to_jshtml())

In [ ]:
data_array = star8.generate_events()

transform = tonic.transforms.ToFrame(
    sensor_size=(star8.image_size[1], star8.image_size[0], 2),
    time_window=10,
    # event_count=500,
    overlap=0.5,
)
anim = animate_events(data_array, transform, fig_size=(5,5))
anim.save(filename="event_star8.gif", writer="pillow")
HTML(anim.to_jshtml())

In [ ]:
sample = data_array[(data_array['t'] >= 200) & (data_array['t'] <= 201)]
plt.quiver(sample['x'], sample['y'], 20* sample['v_x'], 20*sample['v_y'], 
           color=np.where(sample['p'], 'r', 'b'), angles='xy', scale_units='xy', scale=1)
plt.gca().invert_yaxis()
plt.axis('equal')
plt.title("Optical Flow of Events at t = 200-201 mS")
plt.savefig('optical_flow_at_200.png', dpi=300)
# plt.xlim(0, img_width)
# plt.ylim(0, img_height)
plt.show()

In [ ]:
data_array = star8.generate_events()
data = numpy2pyg_event_convertor(data_array).clone()
data['v'] = torch.tensor([data_array['v_x'], data_array['v_y']]).T

In [ ]:
filter_size = 5
tau = 1

In [ ]:
harris_rec = HarrisRecursive(tau, filter_size, image_size)
harris_rec(data_array)
harris_eig1 = harris_rec.eig1
harris_eig2 = harris_rec.eig2
filter_values = harris_rec.filter_value_recursive

In [ ]:
idx = (data.pos[..., -1] > 200) & (data.pos[..., -1] < 210)
scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=harris_eig1[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('eig1')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'filter_values.png'), dpi=300)
# Displaying the plot
plt.show()

scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=harris_eig2[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('eig2')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'filter_values.png'), dpi=300)
# Displaying the plot
plt.show()

scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=filter_values[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('filter values')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'filter_values.png'), dpi=300)
# Displaying the plot
plt.show()

scatter = plt.scatter(data.pos[idx,0], data.pos[idx,1], c=v_mag[idx], cmap='viridis', marker='.', s=5)  # `c` maps color, `s` controls size
plt.colorbar(scatter, label='Value')
plt.xlabel('x [pixels]')
plt.ylabel('y [pixels]')
plt.title('Magnitude of Velocity')
plt.axis('equal')
# plt.savefig(osp.join(saving_folder, f'magnitude_velocity.png'), dpi=300)
# Displaying the plot
plt.show()

sample = data_array[idx]
plt.quiver(sample['x'], sample['y'], 10* sample['v_x'], 10*sample['v_y'], 
           color=np.where(sample['p'], 'r', 'b'), angles='xy', scale_units='xy', scale=1)
plt.axis('equal')
plt.title("Optical Flow of Events")
# plt.savefig(osp.join(saving_folder, f'optical_flow_directions.png'), dpi=300)
plt.show()

In [ ]:
import torch

pos = data.pos  # shape [num_nodes, num_features]

# Compute pairwise distances
dist = torch.cdist(pos, pos)  # [N, N]


In [ ]:
k = 5  # number of neighbors (including self)
# Get the indices of k nearest neighbors for each node
knn_idx = dist.topk(k, largest=False).indices  # [N, k]
print(knn_idx[:10,:])

In [ ]:
def build_knn_features(X, knn_idx):
    """
    Expand input features using neighbors.
    Args:
        X: [N, F] feature matrix
        knn_idx: [N, k] indices of neighbors for each row
    Returns:
        knn_features: [N, k*F] expanded features
    """
    N, F = X.shape
    k = knn_idx.shape[1]
    
    # Gather neighbor features
    neighbors = X[knn_idx]  # shape [N, k, F]
    
    # Flatten neighbors for each row
    knn_features = neighbors.reshape(N, k * F)
    return knn_features

In [ ]:
# Combine inputs and outputs
X = np.stack([data.pos[:,0], data.pos[:,1], harris_eig1, harris_eig2, filter_values], axis=1)  # shape (N, 4)
print("X shape:", X.shape)
Y = data.v
X_with_neighbors = build_knn_features(X, knn_idx)
print("X_with_neighbors shape:", X_with_neighbors.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X_with_neighbors, Y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
Y_train = torch.tensor(Y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.float32)

In [ ]:
X_with_neighbors.shape

In [ ]:
X_train[:10,:]

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        return self.net(x)

input_dim = X_train.size(1)
model = MLP(input_dim=input_dim, hidden_dim=64, output_dim=2)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 100
batch_size = 16

for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(X_train.size(0))
    
    epoch_loss = 0
    for i in range(0, X_train.size(0), batch_size):
        idx = permutation[i:i+batch_size]
        batch_x = X_train[idx]
        batch_y = Y_train[idx]
        
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_x.size(0)
    
    epoch_loss /= X_train.size(0)
    
    # Validation
    model.eval()
    with torch.no_grad():
        preds_test = model(X_test)
        mse_test = criterion(preds_test, Y_test).item()
    
    print(f"Epoch {epoch+1:03d} | Train Loss: {epoch_loss:.4f} | Test MSE: {mse_test:.4f}")



In [ ]:
mse = np.sum((preds_test.numpy() - Y_test.numpy())**2, axis=0)/(preds_test.shape[0])
print(mse)
print(mse.mean())

In [ ]:
mse = np.sum((Y_test.numpy().mean(axis=0) - Y_test.numpy())**2, axis=0)/(Y_test.shape[0])
print(mse)
print(mse.mean())

In [ ]:
mse = np.sum((preds_test.numpy().mean(axis=0) - preds_test.numpy())**2, axis=0)/(preds_test.shape[0])
print(mse)
print(mse.mean())